# DR-OPIC Tutorial

This notebook explains DR-OPIC end to end: the idea, the math, the Python API, CLI usage, and the artifact files.

DR-OPIC means Domain-Routed On-Policy Iterative Correction. It is for coding SLM experiments where the student model attempts first, executable verifiers expose failures, repairs are verified, and training records are built from the student's reachable failure states.

This notebook uses tiny deterministic examples only. It does not require private datasets or model weights.


## 1. Install

From the repository root:

```bash
python -m pip install -e ".[dev]"
python -m pytest -q
```

The examples below assume the package is importable as `dr_opic`.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from dr_opic.maths import smoothed_pass_rate, zpd_weight, group_relative_advantages
from dr_opic.verifier import PythonTask, verify_python, static_check_python
from dr_opic.schemas import Task, Candidate
from dr_opic.forge import rollout_python_task, repair_failures, build_round_artifacts, save_round_artifacts
from dr_opic.delta import build_delta_example
from dr_opic.scheduler import schedule_group, training_mix
from dr_opic.routing import route_task
from dr_opic.safety import classify_coding_safety, selective_accept
from dr_opic.compression import estimate_dense_model
from dr_opic.datasets import audit_rows


## 2. Core Loop

DR-OPIC does this:

1. Route a task to a bounded coding domain, or abstain.
2. Let the current student attempt the task before using any teacher answer.
3. Run executable verification.
4. Compute task learnability with ZPD weighting.
5. Repair failed attempts using failed code plus verifier observation.
6. Verify repairs.
7. Select the most learnable verified winner.
8. Emit JSON and JSONL artifacts for training and review.


## 3. ZPD Math

For `s` passes in `K` samples, use Jeffreys smoothing:

```text
p_tilde = (s + 0.5) / (K + 1)
```

The zone-of-proximal-development weight is:

```text
w_zpd = 4 * p_tilde * (1 - p_tilde)
```

This peaks when a task is neither mastered nor impossible for the current student.


In [ ]:
for passes, samples in [(0, 4), (1, 4), (2, 4), (4, 4)]:
    print({
        "passes": passes,
        "samples": samples,
        "p_tilde": round(smoothed_pass_rate(passes, samples), 4),
        "zpd_weight": round(zpd_weight(passes, samples), 4),
    })


## 4. Verifier Reward

The framework ranks candidates with a scalar reward:

```text
reward =
  1.00 * final_pass
+ 0.25 * public_test_fraction
+ 0.10 * syntax_ok
+ 0.05 * import_ok
- penalties
```

Final pass dominates. Partial reward is for ranking and scheduling, not for pretending a failed candidate passed.


## 5. Verify Python Code

`verify_python` extracts code, runs static checks, writes a temporary candidate file, executes tests in a subprocess, and returns a structured result.

Important: this is a research helper, not a security sandbox. Use a container or VM for untrusted model output.


In [ ]:
py_task = PythonTask(
    prompt="Implement reverse_words(s) returning words in reverse order.",
    entrypoint="reverse_words",
    tests="""assert reverse_words('one two three') == 'three two one'
assert reverse_words('solo') == 'solo'""",
)

bad_code = """def reverse_words(s):
    return s
"""
good_code = """def reverse_words(s):
    return ' '.join(reversed(s.split()))
"""

print("bad:", verify_python(bad_code, py_task))
print("good:", verify_python(good_code, py_task))


In [ ]:
unsafe = """import os
def run():
    os.system('echo no')
"""
static_check_python(unsafe, entrypoint="run")


## 6. Student-First Forge Round

A generator callback stands in for the current student model. A repair callback stands in for a teacher, repair model, or deterministic repair policy.


In [ ]:
task = Task(
    task_id="reverse_words_demo",
    prompt="Implement reverse_words(s) returning words in reverse order.",
    entrypoint="reverse_words",
    tests="""assert reverse_words('one two three') == 'three two one'
assert reverse_words('solo') == 'solo'""",
)

def student(task: Task, index: int) -> str:
    if index == 0:
        return """def reverse_words(s):
    return s
"""
    return """def reverse_words(s):
    return ' '.join(reversed(s.split()))
"""

def repair(task: Task, failed: Candidate, round_index: int) -> str:
    return """def reverse_words(s):
    return ' '.join(reversed(s.split()))
"""

group = rollout_python_task(task, student, k=2)
repairs = repair_failures(group, repair, rounds=1)
artifacts = build_round_artifacts(group, repairs)

artifacts.keys(), artifacts["passes"], artifacts["zpd_weight"], artifacts["winner"]["source"]


In [ ]:
artifacts["winner"]


## 7. Group-Relative Advantages

For rollout rewards `r_i`:

```text
A_i = (r_i - mean(r)) / (std(r) + eps)
```

A low-compute self-training objective can behavior-clone only positive-advantage samples:

```text
L_self = -w_task * sum_i max(A_i, 0) * log pi_theta(y_i | x)
```


In [ ]:
rewards = [row["reward"] for row in artifacts["rollouts"]]
group_relative_advantages(rewards)


## 8. Learnable Winner Selection

For a verified candidate `c` and failed student attempt `s`:

```text
score(c, s) =
  pass_bonus
+ fuzz_bonus
+ model_logprob_bonus
- edit_distance_penalty
- complexity_penalty
- dependency_penalty
- length_penalty
```

The goal is not the prettiest teacher answer. The goal is the verified fix closest to the student's reachable failure state.


## 9. Counterfactual Delta-Span Training

Whole-answer negative training can punish correct shared code. Delta-span training compares failed code and fixed code.

```text
D_plus  = fixed-code tokens touched by insert or replace
D_minus = failed-code tokens touched by delete or replace
```

The training idea:

```text
increase probability on D_plus
subtract only D_minus relative to a reference model
leave shared correct code alone
```


In [ ]:
delta_example = build_delta_example("reverse_words_demo", bad_code, good_code)
delta_example.to_dict()


## 10. Verifier-ZPD Scheduler

The scheduler turns rollout evidence into a curriculum decision.

Buckets:

- `mastered`: already high pass rate
- `zpd_train`: useful student-on-policy training
- `repair_train`: student failed, but a close verified repair exists
- `decompose`: too hard without a close repair
- `eval_only`: not a training split
- `discard`: verifier is unreliable


In [ ]:
scheduled = schedule_group(group, repairs)
scheduled, training_mix([scheduled])


## 11. Save Artifacts

A professional run should save stable artifact files:

- `round_summary.json`
- `student_rollouts.jsonl`
- `verified_repairs.jsonl`
- `learnable_winner.json`
- `delta_spans.json`


In [ ]:
with TemporaryDirectory() as tmp:
    paths = save_round_artifacts(tmp, artifacts)
    for name, path in paths.items():
        print(name, Path(path).name, Path(path).stat().st_size)


## 12. Routing, Safety, And Abstention

A specialist should not answer every prompt. Route first, then apply safety checks, then accept or abstain.


In [ ]:
prompt = "Fix this Python traceback and add pytest coverage."
route = route_task(prompt)
safety = classify_coding_safety(prompt)
print(route)
print(safety)
print("accepted:", selective_accept(route.accepted, safety.allowed, route.confidence))


## 13. Dataset Audits

The repo does not include training data. The audit helper validates external rows before training.

SFT schema:

```json
{"prompt": "...", "response": "..."}
```

Preference rows must include `prompt`, `chosen`, and `rejected`.


In [ ]:
rows = [
    {"prompt": "Implement inc(x) for integers.", "response": """def inc(x):
    return x + 1
"""},
]
audit_rows(rows, schema="sft")


## 14. Cost And Compression Estimates

For a dense transformer with `N` parameters:

```text
training_flops ~= 6 * N * training_tokens
inference_flops_per_token ~= 2 * N
weight_memory ~= bytes_per_param * N
```

Compression is a release phase after behavior improves. It should not be counted as capability by itself.


In [ ]:
estimate_dense_model(3.09e9)


## 15. CLI Cheatsheet

Run from the repository root:

```bash
python -m dr_opic.cli forge-demo
python -m dr_opic.cli --output outputs/demo forge-demo
python -m dr_opic.cli verify-python examples/python_task.json --code examples/reverse_words_good.py
python -m dr_opic.cli verify-python examples/python_task.json --code examples/reverse_words_bad.py --fail-on-error
python -m dr_opic.cli zpd --passes 2 --samples 5
python -m dr_opic.cli route "Fix this Python traceback and add pytest coverage"
python -m dr_opic.cli estimate-model --params 3.09e9
python -m dr_opic.cli delta --task-id reverse_words --failed examples/reverse_words_bad.py --fixed examples/reverse_words_good.py
python -m dr_opic.cli schedule-demo
python -m dr_opic.cli audit-jsonl C:/datasets/slm/sft.jsonl --schema sft
```


## 16. Release Checklist

Do not promote a run just because one number moved. A serious DR-OPIC release should report:

- greedy@1
- coverage@K
- selected@K
- selector gap
- repair@1
- hard-subset pass rate
- malformed output rate
- repeated-token collapse rate
- OOD false accept rate
- unsafe compliance rate
- latency and memory
- contamination and dataset audit summary

The falsifiable claim: student-first repair and delta-span training should improve selected@K or repair@1 without damaging formatting, safety, or abstention.
